In [1]:
import tensorflow as tf

In [2]:
pip install -q -U keras-tuner

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.1/129.1 kB 4.3 MB/s eta 0:00:00


In [3]:
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

(X_train.shape, y_train.shape), (X_test.shape, y_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


(((60000, 28, 28), (60000,)), ((10000, 28, 28), (10000,)))

In [4]:
X_train = X_train / 255.0
X_test = X_test / 255.0

X_train.min(), X_train.max()

(0.0, 1.0)

In [5]:
def model_builder(hp):
  model = tf.keras.Sequential()
  model.add(tf.keras.layers.Flatten(input_shape=(28, 28)))

  hp_activation = hp.Choice('activation', values=['relu', 'tanh'])
  hp_layer_1 = hp.Int('layer_1', min_value=1, max_value=1000, step=100)
  hp_layer_2 = hp.Int('layer_2', min_value=1, max_value=1000, step=100)
  hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

  model.add(tf.keras.layers.Dense(units=hp_layer_1, activation=hp_activation))
  model.add(tf.keras.layers.Dense(units=hp_layer_2, activation=hp_activation))
  model.add(tf.keras.layers.Dense(10, activation='softmax'))

  model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=hp_learning_rate),
                loss=tf.keras.losses.SparseCategoricalCrossentropy(),
                metrics=['accuracy'])

  return model

In [6]:
import keras_tuner as kt

tuner = kt.Hyperband(model_builder,
                     objective='val_accuracy',
                     max_epochs=10,
                     factor=3,
                     directory='dir',
                     project_name='x')

/usr/local/lib/python3.11/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [7]:
stop_early = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

In [8]:
tuner.search(X_train, y_train, epochs=50, validation_split=0.2, callbacks=[stop_early])

Trial 30 Complete [00h 00m 40s]
val_accuracy: 0.9743333458900452

Best val_accuracy So Far: 0.9801666736602783
Total elapsed time: 00h 11m 31s


In [9]:
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

In [10]:
model = tuner.hypermodel.build(best_hps)
history = model.fit(X_train, y_train, epochs=50, validation_split=0.2,
                    callbacks=[stop_early])

Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.8428 - loss: 0.6156 - val_accuracy: 0.9527 - val_loss: 0.1689
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9544 - loss: 0.1601 - val_accuracy: 0.9648 - val_loss: 0.1227
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9719 - loss: 0.0984 - val_accuracy: 0.9693 - val_loss: 0.1002
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9802 - loss: 0.0696 - val_accuracy: 0.9733 - val_loss: 0.0912
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 6s 3ms/step - accuracy: 0.9856 - loss: 0.0500 - val_accuracy: 0.9767 - val_loss: 0.0809
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9902 - loss: 0.0372 - val_accuracy: 0.9762 - val_loss: 0.0813
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9935 - loss: 0.0268 - val_accuracy: 0.9762 - val_loss: 0.0781
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.9962 - loss: 0.0184 - 

In [11]:
history.history

{'accuracy': [0.9081666469573975,
  0.9590416550636292,
  0.9725833535194397,
  0.9799791574478149,
  0.9851666688919067,
  0.9893749952316284,
  0.9925416707992554,
  0.9947916865348816,
  0.9960416555404663,
  0.9973958134651184],
 'loss': [0.34331631660461426,
  0.141397163271904,
  0.0951119214296341,
  0.06850927323102951,
  0.05113575607538223,
  0.03815078362822533,
  0.028085703030228615,
  0.02109396643936634,
  0.016031406819820404,
  0.011614949442446232],
 'val_accuracy': [0.9526666402816772,
  0.9648333191871643,
  0.9692500233650208,
  0.9733333587646484,
  0.9766666889190674,
  0.9761666655540466,
  0.9762499928474426,
  0.9770833253860474,
  0.9776666760444641,
  0.9769166707992554],
 'val_loss': [0.1689366102218628,
  0.12269297242164612,
  0.10017144680023193,
  0.09121806174516678,
  0.08088137954473495,
  0.08133956789970398,
  0.07810520380735397,
  0.08052023500204086,
  0.0802220031619072,
  0.08421152085065842]}

In [12]:
import pandas as pd

pd.DataFrame(history.history)

,accuracy,loss,val_accuracy,val_loss
0,0.908167,0.343316,0.952667,0.168937
1,0.959042,0.141397,0.964833,0.122693
2,0.972583,0.095112,0.969250,0.100171
3,0.979979,0.068509,0.973333,0.091218
4,0.985167,0.051136,0.976667,0.080881
5,0.989375,0.038151,0.976167,0.081340
6,0.992542,0.028086,0.976250,0.078105
7,0.994792,0.021094,0.977083,0.080520
8,0.996042,0.016031,0.977667,0.080222
9,0.997396,0.011615,0.976917,0.084212


In [13]:
model

<Sequential name=sequential_1, built=True>